# Sala Chaturamuk Phaichit — Gaussian Splatting, 352 photographs (Kaggle)

Same pipeline as the Colab notebook, rewired for Kaggle. The reason to be here: **Save & Run All
commits the notebook to Kaggle's own servers and runs it detached.** Closing the laptop, losing
wifi, or shutting the browser has no effect. The whole pipeline fits inside one 12-hour session,
so in the normal case nothing needs to resume at all.

## Setup, in order

1. **Upload the photographs once, as a Dataset.** Left sidebar → *Datasets* → *New Dataset* →
   upload `sala_352.zip`. Name it `sala-352`. It persists forever; you never upload again.
2. **Attach it:** right sidebar → *Input* → *Add Input* → `sala-352`.
3. **Accelerator → GPU T4 ×2.** Not the P100 (compute 6.0 — the extensions build and then
   training aborts). Note the second T4 sits idle: 3DGS is single-GPU. T4 ×2 is simply the
   healthy Turing option.
4. **Internet → On.** Without it `apt-get`, `git clone` and `pip install` all fail. This is the
   single most common reason the run dies at step 4.
5. **Save Version → Save & Run All (Commit).** Do not babysit it interactively.

## Expected wall time

Kaggle gives 4 CPU cores against Colab's 2, so the COLMAP stages roughly halve.

| Stage | Kaggle estimate | Basis |
|---|---|---|
| Unpack + downscale 352 images | 4 min | measured on Colab, CPU-bound |
| COLMAP feature extraction | 12–15 min | 25 min on Colab's 2 cores |
| COLMAP sequential matching | 25–35 min | 45–60 min on Colab |
| COLMAP mapper | 10–20 min | mostly single-threaded, scales least |
| Undistort + sky masking | 8–10 min | |
| Compile the CUDA extensions | 5–8 min | measured, previous Kaggle run |
| **Training, 30 000 iterations** | **2.5–3 h** | 1 h 36 min measured for 146 images; 351 here |
| Render + score held-out views | 5 min | |
| **Total** | **≈ 4–5 h** | inside the 12 h session limit |

The training figure is the one extrapolated rather than measured: 1 h 36 min is a real number from
your `sala-v2-pinhole` run at 146 images, and cost scales with Gaussian count rather than image
count, so 2.5–3 h is an estimate, not a measurement.

## Resuming, if it ever comes to that

Every expensive stage still banks into `/kaggle/working/out`, which becomes the notebook version's
**Output**. To resume a later run: *Add Input* → *Notebook Output* → this notebook's last version.
Every cell searches `/kaggle/input` as well as the current output, so it skips whatever that
version already banked — and re-emits it, so you only ever attach the **one latest** version
rather than accumulating a chain of them.

Scratch lives in `/kaggle/temp` (wiped, uncounted); only deliverables go in `/kaggle/working`,
which is capped at 20 GB.

## 1 · GPU check

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Right sidebar -> Accelerator -> GPU T4 x2.'

# The P100 compiles the extensions happily and only fails once training touches
# the GPU, which is a confusing way to lose an hour. Catch it here instead.
major, _ = torch.cuda.get_device_capability(0)
name = torch.cuda.get_device_name(0)
print('device:', name, f'(compute {major}.x)')
assert major >= 7, (f'{name} is compute capability {major}.x, which this PyTorch build '
                    'does not support. Switch the accelerator to GPU T4 x2.')

## 2 · Paths, photographs, and where we left off

No Drive here. Scratch goes to `/kaggle/temp` (not saved, not counted against the 20 GB output
cap); anything worth keeping goes to `/kaggle/working/out`.

`cached()` looks in this session's output **first**, then across every attached input — so a
previous version's output attached as an input is picked up automatically.

In [ ]:
import os, glob, zipfile, shutil, time

# ======================= run settings ====================================
# Stages listed in FORCE ignore any cached copy and are recomputed, so a
# parameter can be changed without starting over from nothing.
#   run 1  FORCE = set()                  loop_detection 1  -> v1 55.3  v2 29.2  folded
#   run 2  FORCE = {'matching','mapper'}  loop_detection 0  -> v1 27.4  v2  5.4  v2 clean
#   run 3  FORCE = set()                  reuse run 2's poses; only the CHECK was wrong
FORCE = set()
GATE  = {'v2 '}        # Only captures whose ordering is VERIFIED continuous may stop
                       # the run. v2 has EXIF on all 146 frames, one session, median
                       # 3.0 s apart, so a high step ratio there really does mean a
                       # fold. v1a has no EXIF at all: a high ratio there is unreadable
                       # - it could be a fold, or a capture that was never one walk.
                       # Those are reported every run, and never gate it.
LOOP_DETECTION = 0     # vocab-tree loop closure. Zero on a four-fold symmetric
                       # subject: it is the thing that pairs views 90 deg apart.
# =========================================================================

WORK  = '/kaggle/temp/work'          # scratch: wiped, uncounted
OUT   = '/kaggle/working/out'        # deliverables: become the notebook Output
SCENE = f'{WORK}/scene'
SRC   = f'{SCENE}/input'
DB    = f'{SCENE}/database.db'
MODEL_DIR = f'{OUT}/sala_352_masked'
for d in (WORK, OUT, SCENE): os.makedirs(d, exist_ok=True)

STAGE_OF = {'db_extracted.db': 'extraction',
            'db_matched.db':   'matching',
            'sparse':          'mapper'}

def cached(name):
    # Banked artifact from this session's output, else from any attached input.
    if STAGE_OF.get(name) in FORCE:
        print(f'  [FORCE] ignoring any cached {name}')
        return None
    p = f'{OUT}/{name}'
    if os.path.exists(p): return p
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True))
    return hits[0] if hits else None

def adopt(hit, name):
    # Copy a restored artifact into this run's output as well. Without this, a
    # version that skipped a stage would not re-emit it, and a third run would
    # need every earlier version attached instead of just the latest one.
    dst = f'{OUT}/{name}'
    if os.path.abspath(hit) != os.path.abspath(dst):
        if os.path.isdir(hit):
            if os.path.exists(dst): shutil.rmtree(dst)
            shutil.copytree(hit, dst)
        else:
            shutil.copy(hit, dst)
    return dst

# Kaggle EXTRACTS a .zip when a dataset is uploaded, so the photographs may
# arrive either as the archive or as plain folders. Handle both.
z = sorted(glob.glob('/kaggle/input/**/sala_352.zip', recursive=True))
if z:
    print('unpacking', z[0])
    with zipfile.ZipFile(z[0]) as f: f.extractall(f'{WORK}/raw')
    RAW = f'{WORK}/raw'
else:
    RAW = '/kaggle/input'
    print('no sala_352.zip - assuming Kaggle auto-extracted the dataset')

# Filter by the known capture folders so an attached previous-version output
# (renders, checkpoints) cannot be mistaken for input photographs.
allj = glob.glob(f'{RAW}/**/*.jpg', recursive=True)
imgs = sorted(p for p in allj if 'salathai_version' in p) or sorted(allj)

print('photos  :', RAW)
print('scratch :', WORK)
print('outputs :', OUT)
print()
print('--- resume report ---')
for label, nm in (('extraction', 'db_extracted.db'),
                  ('matching  ', 'db_matched.db'),
                  ('mapper    ', 'sparse')):
    hit = cached(nm)
    print(f'  {label} cached : {bool(hit)}' + (f'   <- {hit}' if hit else ''))
ck = sorted(glob.glob(f'{OUT}/**/chkpnt*.pth', recursive=True)
            + glob.glob('/kaggle/input/**/chkpnt*.pth', recursive=True),
            key=lambda p: int(''.join(c for c in os.path.basename(p) if c.isdigit())))
print(f'  training ckpts    : {[os.path.basename(c) for c in ck] if ck else "none"}')

print(f'\n{len(imgs)} photographs found')
assert len(imgs) > 300, (f'expected 352, found {len(imgs)}. Right sidebar -> Input '
                         '-> Add Input -> your sala-352 dataset.')

## 3 · Downscale to 1600 px

In [ ]:
import cv2
os.makedirs(SRC, exist_ok=True)
if len(glob.glob(f'{SRC}/*.jpg')) >= 300:
    print('already downscaled, skipping')
else:
    def tag(p):
        low = p.lower()
        return 'a_v1' if 'version1' in low else ('b_v2' if 'version2' in low else 'c_xx')
    groups = {}
    for p in imgs: groups.setdefault(tag(p), []).append(p)
    for g in groups: groups[g].sort()
    for g, ps in sorted(groups.items()):
        for k, p in enumerate(ps):
            im = cv2.imread(p)
            if im is None: continue
            h, w = im.shape[:2]
            if w > 1600: im = cv2.resize(im, (1600, round(h*1600/w)), interpolation=cv2.INTER_AREA)
            cv2.imwrite(f'{SRC}/{g}_{k:04d}.jpg', im, [cv2.IMWRITE_JPEG_QUALITY, 95])
        print(f'{g}: {len(ps)}')
print('images ready:', len(glob.glob(f'{SRC}/*.jpg')))

## 4 · Install COLMAP

Needs *Internet → On*. Kaggle's image is Ubuntu-based so the distribution package works, but
unlike the training half of this notebook **this step has never run on Kaggle before** — the
previous Kaggle run consumed poses computed locally. If `apt-get` cannot find `colmap`, that is
what has gone wrong, and the fallback in the next comment is the fix.

In [ ]:
# Kaggle is headless. COLMAP links Qt, whose default plugin wants an X display,
# so without this every colmap call aborts before doing any work.
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

if shutil.which('colmap') is None:
    !apt-get -qq update > /dev/null 2>&1
    !apt-get -qq install -y colmap > /dev/null 2>&1

# Fallback if the distro package is unavailable:
#   !apt-get -qq install -y software-properties-common
#   !add-apt-repository -y ppa:ubuntugis/ppa && apt-get -qq update && apt-get -qq install -y colmap
assert shutil.which('colmap'), ('colmap did not install. Check Internet is On in the right '
                                'sidebar, then try the add-apt-repository fallback above.')
print(shutil.which('colmap'))
!colmap -h 2>&1 | head -3

## 5 · Feature extraction — cached

GPU SIFT needs an OpenGL context, which a headless session cannot provide, so this runs on CPU.
Kaggle's 4 cores make it roughly twice as fast as Colab's 2.

In [ ]:
hit_m, hit_e = cached('db_matched.db'), cached('db_extracted.db')
if hit_m:
    shutil.copy(adopt(hit_m, 'db_matched.db'), DB)
    print('matched database restored - extraction and matching both skipped')
elif hit_e:
    shutil.copy(adopt(hit_e, 'db_extracted.db'), DB)
    print('extracted database restored - skipping extraction')
else:
    !rm -f {DB}
    t0 = time.time()
    !QT_QPA_PLATFORM=offscreen colmap feature_extractor \
        --database_path {DB} --image_path {SRC} \
        --ImageReader.single_camera 1 \
        --ImageReader.camera_model SIMPLE_RADIAL \
        --SiftExtraction.use_gpu 0 \
        --SiftExtraction.max_image_size 1600 \
        --SiftExtraction.max_num_features 8192
    assert os.path.exists(DB), 'extraction produced no database'
    shutil.copy(DB, f'{OUT}/db_extracted.db')
    print(f'extraction done in {(time.time()-t0)/60:.1f} min, banked to output')

## 6 · Matching — cached

The pavilion is four-faced, so views 90 degrees apart look alike; exhaustive matching pairs up
different faces and the mapper folds the geometry. Sequential matching avoids that, and loop
detection bridges the join between the two separate walks.

In [ ]:
if cached('db_matched.db'):
    print('matching already cached - skipping')
else:
    t0 = time.time()
    cmd = ('QT_QPA_PLATFORM=offscreen colmap sequential_matcher'
           f' --database_path {DB}'
           ' --SiftMatching.use_gpu 0'
           ' --SiftMatching.max_num_matches 16384'
           ' --SequentialMatching.overlap 8'
           f' --SequentialMatching.loop_detection {LOOP_DETECTION}')
    if LOOP_DETECTION:
        VOCAB = f'{WORK}/vocab_tree_flickr100K_words32K.bin'
        if not os.path.exists(VOCAB):
            !wget -q https://demuc.de/colmap/vocab_tree_flickr100K_words32K.bin -O {VOCAB}
        cmd += f' --SequentialMatching.vocab_tree_path {VOCAB}'
    print(cmd, '\n')
    !{cmd}
    shutil.copy(DB, f'{OUT}/db_matched.db')
    print(f'matching done in {(time.time()-t0)/60:.1f} min, banked to output')

## 7 · Mapper — cached, with snapshots

`snapshot_images_freq` writes a partial reconstruction every 40 registered images, so a mapper
that dies half way leaves something to inspect rather than nothing.

In [ ]:
SPARSE = f'{SCENE}/sparse'
hit_s = cached('sparse')
if hit_s and (glob.glob(f'{hit_s}/*/images.bin') or glob.glob(f'{hit_s}/*/images.txt')):
    if os.path.exists(SPARSE): shutil.rmtree(SPARSE)
    shutil.copytree(adopt(hit_s, 'sparse'), SPARSE)
    print(f'mapper result restored from {hit_s} - skipping')
else:
    os.makedirs(SPARSE, exist_ok=True)
    SNAP = f'{OUT}/mapper_snapshots'; os.makedirs(SNAP, exist_ok=True)
    t0 = time.time()
    !QT_QPA_PLATFORM=offscreen colmap mapper \
        --database_path {DB} --image_path {SRC} --output_path {SPARSE} \
        --Mapper.snapshot_path {SNAP} \
        --Mapper.snapshot_images_freq 40
    assert glob.glob(f'{SPARSE}/*'), 'mapper produced no model'
    dst = f'{OUT}/sparse'
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(SPARSE, dst)
    print(f'mapper done in {(time.time()-t0)/60:.1f} min, banked to output')
!ls {SPARSE}

## 8 · Fold check — read this before training

A walk photographed in order should have consecutive cameras close together. If the geometry has
folded, some consecutive pairs land far apart, and the ratio of largest to median step exposes it
regardless of scale. **Healthy reference: version2 alone scores 7.0.**

What the two attempts so far established, both on all 352 photographs:

| Matching | Registered | Final cost | v1 ratio | v2 ratio |
|---|---|---|---|---|
| exhaustive | 351/352 | 0.99 px | 102.2 | 36.0 |
| sequential + loop detection | 351/352 | 0.66 px | 55.3 | 29.2 |
| *version2 alone* | *146/146* | *0.58 px* | — | *7.0* |

Loop detection halved the damage and did not remove it. **A low reprojection error is not evidence
of correct geometry** — under four-fold symmetry the mapper folds views 90 degrees apart onto each
other and then fits them beautifully. That is a finding for the write-up, not just a nuisance.

With `LOOP_DETECTION = 0` there is no longer any mechanism pairing non-adjacent frames, so a fold
is structurally unlikely. The cost is that the two walks may not link at all — expect COLMAP to
emit **two sub-models**. That is an acceptable outcome: this cell reports every sub-model and then
works with the largest, which would be capture 1 at 206 images.

**This cell raises if the chosen model is folded**, so a detached run stops here rather than
spending three hours training on bad poses.

In [ ]:
import numpy as np

PICK = None          # e.g. '0' to override which sub-model is used

sub = sorted(glob.glob(f'{SPARSE}/*'))
print('sub-models:', [os.path.basename(x) for x in sub])
for m in sub:
    if not os.path.exists(f'{m}/images.txt'):
        !QT_QPA_PLATFORM=offscreen colmap model_converter --input_path {m} --output_path {m} --output_type TXT

def read_pos(model):
    pos = {}
    for line in open(f'{model}/images.txt'):
        if line.startswith('#') or not line.strip(): continue
        f = line.split()
        if len(f) < 10 or not f[0].isdigit(): continue
        qw,qx,qy,qz,tx,ty,tz = map(float, f[1:8]); nm = f[9]
        q = np.array([qw,qx,qy,qz]); q /= np.linalg.norm(q); w,x,y,z = q
        R = np.array([[1-2*(y*y+z*z), 2*(x*y-w*z), 2*(x*z+w*y)],
                      [2*(x*y+w*z), 1-2*(x*x+z*z), 2*(y*z-w*x)],
                      [2*(x*z-w*y), 2*(y*z+w*x), 1-2*(x*x+y*y)]])
        pos[nm] = -R.T @ np.array([tx,ty,tz])
    return pos

# salathai_version1 is NOT one walk: capture_0000-0130 are 4032x3024 with no EXIF,
# capture_0131-0205 are 5712x4284 from an iPhone 17 Pro. Treating all 206 as a single
# ordered walk puts a session seam in the middle of the path and reads it as a fold.
# version2 is one continuous walk (146 photos, one day, median 3.0 s apart).
import re
def _idx(n):
    m = re.search(r'_(\d{4})\.', n)
    return int(m.group(1)) if m else -1

CAPTURES = {
    'v1a': lambda n: n.startswith('a_v1') and _idx(n) <= 130,
    'v1b': lambda n: n.startswith('a_v1') and _idx(n) >= 131,
    'v2 ': lambda n: n.startswith('b_v2'),
}

def fold_ratios(pos):
    out = {}
    for pre, belongs in CAPTURES.items():
        ns = sorted(n for n in pos if belongs(n))
        if len(ns) < 5: continue
        P = np.array([pos[n] for n in ns])
        step = np.linalg.norm(np.diff(P, axis=0), axis=1)
        med = np.median(step)
        out[pre] = (len(ns), step.max()/med, int((step > 5*med).sum()))
    return out

# Report every sub-model. A split reconstruction is informative, not a failure:
# one clean single-capture model beats one folded combined model.
report = {}
for m in sub:
    p = read_pos(m); report[m] = (p, fold_ratios(p))
    detail = '  '.join(f'{k} {v[0]:3d}img r={v[1]:6.1f}' for k, v in report[m][1].items())
    print(f'  {os.path.basename(m):>3}: {len(p):3d} images   {detail}')

MODEL = f'{SPARSE}/{PICK}' if PICK else max(sub, key=lambda m: len(report[m][0]))
pos = report[MODEL][0]
print(f'\nusing sub-model {os.path.basename(MODEL)}: '
      f'{len(pos)} of {len(os.listdir(SRC))} images\n')

folded = []
for pre, (n, ratio, jumps) in report[MODEL][1].items():
    ok, gated = ratio < 15, pre in GATE
    tag = ('LOOKS OK' if ok else '*** POSSIBLE FOLD ***') if gated else \
          ('looks ok' if ok else 'high, but this capture has no verified ordering')
    print(f'{pre}: {n} imgs  max/median step = {ratio:6.1f}  jumps over 5x = {jumps:3d}  '
          f'{"[GATE]" if gated else "[info]"} {tag}')
    if not ok and gated: folded.append(pre)

assert not folded, (
    f'Fold suspected in {folded}. Healthy reference: version2 alone = 7.0.\n'
    'Levers, in order:\n'
    '  1. LOOP_DETECTION = 0 with FORCE = {"matching","mapper"} in cell 2.\n'
    '  2. Drop the offending capture: it is a whole camera/session, not stray frames.\n'
    '  3. Extraction used --ImageReader.single_camera 1, which forces ONE intrinsic\n'
    '     across three physically different cameras. If a capture keeps folding, that\n'
    '     is the next thing to fix (one camera per capture), and it costs a full re-run.\n'
    'Do not train on folded poses: a low reprojection error does not mean correct geometry.')
print('\nfold check passed')

## 9 · Undistort to a pinhole camera

In [ ]:
PIN = f'{WORK}/scene_pinhole'
if os.path.exists(f'{PIN}/images') and len(glob.glob(f'{PIN}/images/*')) > 300:
    print('already undistorted, skipping')
else:
    !QT_QPA_PLATFORM=offscreen colmap image_undistorter --image_path {SRC} --input_path {MODEL} \
        --output_path {PIN} --output_type COLMAP --max_image_size 1600
print(len(glob.glob(f'{PIN}/images/*')), 'undistorted images')

## 10 · Remove the sky

In the existing 15,000-iteration model, 196,775 of 554,618 Gaussians (35.5%) are bright and
semi-transparent — sky floaters — and the largest 1% by scale sit at a median radius of 9.92
against a scene median of 2.30. Over a third of the model's capacity is painting haze.

The sky is painted **black** rather than carried as alpha. 3DGS renders against black by default,
so a black region costs nothing to explain and the optimiser leaves it empty.

Check the contact sheet in the committed output. A leftover sliver of sky is harmless; a chewed
roofline is not — raise `V_MIN` if edges are being eaten.

In [ ]:
import numpy as np, cv2
V_MIN, S_MAX, ERODE = 140, 50, 9
MARK = f'{PIN}/.sky_removed'

def sky_mask(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV); S, V = hsv[...,1], hsv[...,2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    grad = cv2.magnitude(cv2.Sobel(gray, cv2.CV_32F,1,0,ksize=3),
                         cv2.Sobel(gray, cv2.CV_32F,0,1,ksize=3))
    smooth = cv2.blur((grad < 25).astype(np.uint8), (9,9)) > 0.7
    cand = ((S < S_MAX) & (V > V_MIN) & smooth).astype(np.uint8)
    cand = cv2.morphologyEx(cand, cv2.MORPH_CLOSE, np.ones((7,7), np.uint8))
    _, lab = cv2.connectedComponents(cand)
    top = set(np.unique(lab[: img.shape[0]//20])) - {0}
    return cv2.erode(np.isin(lab, list(top)).astype(np.uint8), np.ones((ERODE,ERODE), np.uint8))

files = sorted(glob.glob(f'{PIN}/images/*'))
if os.path.exists(MARK):
    print('sky already removed, skipping')
else:
    frac = []
    for p in files:
        im = cv2.imread(p); m = sky_mask(im); im[m.astype(bool)] = 0
        cv2.imwrite(p, im, [cv2.IMWRITE_JPEG_QUALITY, 95]); frac.append(m.mean())
    open(MARK,'w').write('done')
    frac = np.array(frac)
    print(f'sky removed: mean {frac.mean()*100:.1f}% per frame '
          f'(min {frac.min()*100:.1f}%, max {frac.max()*100:.1f}%)')

# Contact sheet, saved as a file too - a committed run has no live output to look at.
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
sel = files[::max(1,len(files)//8)][:8]
fig, ax = plt.subplots(2,4, figsize=(16,6))
for a,p in zip(ax.ravel(), sel):
    a.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)); a.axis('off')
    a.set_title(os.path.basename(p), fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/sky_mask_contact_sheet.png', dpi=90); plt.show()
print('contact sheet saved to output')

## 11 · Install 3D Gaussian Splatting

In [ ]:
GS = '/kaggle/working/gaussian-splatting'
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'      # T4; without this the build can guess wrong

# os.chdir rather than %cd: line magics do not reliably expand {GS}, and the
# shell escapes below inherit the kernel's cwd either way.
if not os.path.exists(GS):
    os.chdir('/kaggle/working')
    !git clone --recursive -q https://github.com/graphdeco-inria/gaussian-splatting
    os.chdir(GS)
    !pip install -q plyfile
    !pip install -q submodules/diff-gaussian-rasterization
    !pip install -q submodules/simple-knn
else:
    print('already installed')
os.chdir(GS)
print('cwd:', os.getcwd())

## 12 · Train — resumable

`--checkpoint_iterations` writes a full optimiser state every 5,000 iterations and the cell
restarts from the newest one automatically. On Kaggle this is insurance rather than the plan:
a committed run of ~4–5 h fits inside the 12 h limit, so it should finish in one go.

`--save_iterations` additionally writes viewable `.ply` models at 7k / 15k / 30k.
`--eval` holds out every 8th photograph, so the next cell scores against unseen images.

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)

# Adopt a checkpoint from an attached previous version, if there is one.
for c in sorted(glob.glob('/kaggle/input/**/chkpnt*.pth', recursive=True)):
    dst = f'{MODEL_DIR}/{os.path.basename(c)}'
    if not os.path.exists(dst): shutil.copy(c, dst); print('adopted', os.path.basename(c))

ck = sorted(glob.glob(f'{MODEL_DIR}/chkpnt*.pth'),
            key=lambda p: int(''.join(c for c in os.path.basename(p) if c.isdigit())))
RESUME = ck[-1] if ck else None
print('resuming from', RESUME if RESUME else 'scratch')

os.chdir(GS)
CKPTS = '5000 10000 15000 20000 25000 30000'
if RESUME:
    !python train.py -s {PIN} -m {MODEL_DIR} --iterations 30000 \
        --save_iterations 7000 15000 30000 \
        --checkpoint_iterations {CKPTS} \
        --start_checkpoint {RESUME} --eval
else:
    !python train.py -s {PIN} -m {MODEL_DIR} --iterations 30000 \
        --save_iterations 7000 15000 30000 \
        --checkpoint_iterations {CKPTS} --eval

## 13 · Render the held-out views and score them

In [ ]:
os.chdir(GS)
!python render.py  -m {MODEL_DIR}
!python metrics.py -m {MODEL_DIR}
import json
print(json.dumps(json.load(open(f'{MODEL_DIR}/results.json')), indent=2))
print('\nprevious run, 146 images, no masks:  PSNR 22.09  SSIM 0.785  LPIPS 0.346')

## 14 · Crop the obstacles

Sky masking removes the haze. The trees, lawn and footpath are genuinely photographed and so are
genuinely reconstructed — they come off by cropping in space, the same cut used on the
photogrammetry mesh. Adjust `KEEP_R` from the printed percentiles.

In [ ]:
import numpy as np
from plyfile import PlyData, PlyElement
SRC_PLY = f'{MODEL_DIR}/point_cloud/iteration_30000/point_cloud.ply'
if not os.path.exists(SRC_PLY):
    cands = sorted(glob.glob(f'{MODEL_DIR}/point_cloud/iteration_*/point_cloud.ply'),
                   key=lambda p: int(p.split('iteration_')[1].split('/')[0]))
    assert cands, 'no point_cloud.ply - did training finish?'
    SRC_PLY = cands[-1]
print('using', SRC_PLY)

ply = PlyData.read(SRC_PLY); v = ply['vertex']
xyz = np.stack([v['x'], v['y'], v['z']], 1).astype(np.float64)
C = np.array(list(pos.values()))
centre = np.median(C, axis=0); ring = np.median(np.linalg.norm(C - centre, axis=1))
r = np.linalg.norm(xyz[:, [0,2]] - centre[[0,2]], axis=1)
h = xyz[:,1] - np.median(xyz[:,1])
print(f'{len(xyz):,} gaussians   camera ring radius {ring:.2f}')
for q in (50,70,80,90,95,99): print(f'  radius p{q:<3d} = {np.percentile(r,q):6.2f}')

KEEP_R, KEEP_H = 0.55*ring, 2.5*ring
keep = (r < KEEP_R) & (np.abs(h) < KEEP_H)
print(f'\nkeeping {keep.sum():,} of {len(xyz):,} ({keep.mean()*100:.1f}%) at KEEP_R={KEEP_R:.2f}')

CROPPED = f'{OUT}/sala_352_masked_cropped.ply'
PlyData([PlyElement.describe(v.data[keep], 'vertex')]).write(CROPPED)
print('wrote', CROPPED, f'({os.path.getsize(CROPPED)/1e6:.1f} MB)')

## 15 · Export `.splat` and tidy the output

Last time, the 137 MB `point_cloud.ply` failed to download four times —
`IncompleteRead(3732029 bytes read, 133814766 more expected)`. The `.splat` form of the same model
is 8× smaller (17 MB for 554,618 Gaussians, at 32 bytes each) and drops straight into the viewer
in `viewer/splat.html`. So both are written, and the `.splat` is the one to fetch first.

This cell also deletes the clone and the scene copy from `/kaggle/working` so the committed output
stays small, while keeping the COLMAP caches that make a future run resumable.

In [ ]:
import numpy as np, os, glob, shutil
from plyfile import PlyData

def ply_to_splat(src, dst):
    # 32-byte rows: xyz f32[3], scale f32[3], rgba u8[4], quat u8[4].
    #     matches antimatter15/splat, which viewer/vendor/splat.js is built from.
    p = PlyData.read(src)['vertex']
    SH_C0 = 0.28209479177387814
    xyz    = np.stack([p['x'], p['y'], p['z']], 1).astype(np.float32)
    scales = np.exp(np.stack([p['scale_0'], p['scale_1'], p['scale_2']], 1)).astype(np.float32)
    rot    = np.stack([p['rot_0'], p['rot_1'], p['rot_2'], p['rot_3']], 1).astype(np.float32)
    rot   /= np.linalg.norm(rot, axis=1, keepdims=True)
    rgb    = 0.5 + SH_C0 * np.stack([p['f_dc_0'], p['f_dc_1'], p['f_dc_2']], 1)
    alpha  = 1.0 / (1.0 + np.exp(-np.asarray(p['opacity'], dtype=np.float64)))
    # biggest and most opaque first, so the renderer's progressive sort looks right early
    order  = np.argsort(-(scales.astype(np.float64).prod(1) * alpha))

    n = len(order)
    buf = np.zeros((n, 32), dtype=np.uint8)
    buf[:,  0:12] = xyz[order].view(np.uint8).reshape(n, 12)
    buf[:, 12:24] = scales[order].view(np.uint8).reshape(n, 12)
    buf[:, 24:27] = np.clip(rgb[order] * 255, 0, 255).astype(np.uint8)
    buf[:, 27:28] = np.clip(alpha[order, None] * 255, 0, 255).astype(np.uint8)
    buf[:, 28:32] = np.clip(rot[order] * 128 + 128, 0, 255).astype(np.uint8)
    buf.tofile(dst)
    return n

for src, name in ((SRC_PLY, 'sala_352_masked_full'), (CROPPED, 'sala_352_masked_cropped')):
    dst = f'{OUT}/{name}.splat'
    n = ply_to_splat(src, dst)
    print(f'{name}.splat  {n:,} gaussians  {os.path.getsize(dst)/1e6:.1f} MB')

shutil.copy(SRC_PLY, f'{OUT}/sala_352_masked_full.ply')
for f in ('cameras.json', 'results.json', 'cfg_args', 'per_view.json'):
    p = f'{MODEL_DIR}/{f}'
    if os.path.exists(p): shutil.copy(p, f'{OUT}/{f}')
if os.path.isdir(f'{MODEL_DIR}/test'):
    shutil.make_archive(f'{OUT}/heldout_renders', 'zip', f'{MODEL_DIR}/test')

# Keep only the newest checkpoint: they are ~600 MB each and /kaggle/working caps at 20 GB.
ck = sorted(glob.glob(f'{MODEL_DIR}/chkpnt*.pth'),
            key=lambda p: int(''.join(c for c in os.path.basename(p) if c.isdigit())))
for c in ck[:-1]: os.remove(c); print('dropped', os.path.basename(c))

# The clone is ~1 GB of source and build artifacts and has no business in the
# output. cd out of it first, or the kernel is left with an invalid cwd.
os.chdir('/kaggle/working')
shutil.rmtree(GS, ignore_errors=True)

# save_iterations wrote 7k / 15k / 30k. Keep the last two; 7k is only ever a
# sanity check and is another 130 MB.
for d in sorted(glob.glob(f'{MODEL_DIR}/point_cloud/iteration_*'),
                key=lambda p: int(p.rsplit('_', 1)[1]))[:-2]:
    shutil.rmtree(d, ignore_errors=True); print('dropped', os.path.basename(d))

# Mapper snapshots were insurance against a mapper that died. It did not.
for d in glob.glob(f'{OUT}/mapper_snapshots/*'): shutil.rmtree(d, ignore_errors=True)

total = 0
print('\n--- committed output ---')
for root, _, fs in os.walk('/kaggle/working'):
    for f in sorted(fs):
        p = os.path.join(root, f); sz = os.path.getsize(p); total += sz
        if sz > 1e6: print(f'  {os.path.relpath(p, "/kaggle/working"):55s} {sz/1e6:8.1f} MB')
print(f'\ntotal {total/1e6:.0f} MB of the 20 GB cap')

## What to download, and in what order

From the notebook version's *Output* tab:

1. **`out/sala_352_masked_cropped.splat`** — a few MB. This is the demo model; drop it into
   `viewer/splat.html`.
2. **`out/results.json`** — the held-out PSNR / SSIM / LPIPS for the write-up.
3. **`out/sky_mask_contact_sheet.png`** — check the masking did not chew the roofline.
4. **`out/heldout_renders.zip`** — render-versus-photograph pairs, the paper figures.
5. **`out/sala_352_masked_full.ply`** — 130 MB+, only if SuperSplat editing is wanted. This is the
   one that failed to download repeatedly last time. Prefer `kaggle kernels output` over the
   browser, and if it truncates again, the `.splat` carries the same geometry.

Keep `out/db_matched.db` and `out/sparse/` attached as an input to any future version and the
COLMAP hour never runs again.